# NB-deep2 — gated_full vs unet_drop on the hard splits

Re-evaluates the injection checkpoints (`gated_full`, `gated_nocons`, `gated_nofilm`)
and the baselines (`unet_drop`, `unet_naive`, `manet_drop`) on:
- **Bolivia holdout** (out-of-region) and **flood-only IoU** (permanent water excluded)

The question: does gated_full's SAR-only robustness advantage over naive concat
**widen** on the harder out-of-region split? If yes, the robustness claim is airtight.

No retraining. Auto-detects each checkpoint's type and rebuilds it. ~10 min.

**Attach:** `lgmf_c` (v7), `lgmf-index`, `lgmf-inject` (saved ckpt_inject),
`lgmf-baselines` (saved ckpt_base). GPU on; Internet **off** (no pretrained download).


In [ ]:
import sys, os, glob, json, dataclasses, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v7"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.model import build_model
from lgmf.baselines import build_baseline, build_gated_unet
from lgmf.engine import evaluate
print("lgmf", lgmf.__version__)

idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))
ckpts = glob.glob("/kaggle/input/**/best.pt", recursive=True)
print(f"index: {idx}\ncheckpoints found: {len(ckpts)}")

## Loaders + checkpoint-type dispatch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; regs = ["both", "s1_only", "s2_only"]

base = LGMFConfig()
base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]
base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]
base.num_workers = 0
def loader(split):
    ds = Sen1Floods11(base, f"{idx}/{split}_index.json", augment=False)
    return DataLoader(ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=pin), len(ds)
val_ld, n_val = loader("val"); bol_ld, n_bol = loader("bolivia")
print(f"val {n_val} | bolivia {n_bol}")

FIELDS = {f.name for f in dataclasses.fields(LGMFConfig)}
def build_from_ckpt(ck):
    if ck.get("gated"):
        kw = {k: v for k, v in ck["gated"].items() if k != "out_ch"}
        m = build_gated_unet(pretrained=False, out_ch=ck["gated"].get("out_ch", 8), **kw)
    elif ck.get("baseline"):
        kw = {k: v for k, v in ck["baseline"].items() if k != "in_ch"}
        m = build_baseline(pretrained=False, in_ch=ck["baseline"].get("in_ch", 8), **kw)
    else:
        c = LGMFConfig(**{k: v for k, v in ck["cfg"].items() if k in FIELDS})
        c.s2_bands = tuple(c.s2_bands); c.encoder_pretrained = False
        m = build_model(c)
    m.load_state_dict(ck.get("ema") or ck["model"])
    return m

## Evaluate every checkpoint on val + bolivia

In [ ]:
def parse(p):
    sd = os.path.dirname(p)
    return os.path.basename(os.path.dirname(sd)), os.path.basename(sd)

records = {}
for path in sorted(ckpts):
    cfgname, seed = parse(path)
    ck = torch.load(path, map_location=device, weights_only=False)
    model = build_from_ckpt(ck).to(device).eval()
    val = evaluate(model, val_ld, base, device)
    bol = evaluate(model, bol_ld, base, device)
    def pack(res): return {r: {"iou": res[r]["iou"], "flood": res[r].get("flood_only_iou")} for r in regs}
    records.setdefault(cfgname, []).append({"seed": seed, "val": pack(val), "bol": pack(bol)})
    print(f"  {cfgname:12s} {seed:7s} | val s1 {val['s1_only']['iou']:.3f} | bol s1 {bol['s1_only']['iou']:.3f}")
print("evaluated", sum(len(v) for v in records.values()), "checkpoints")

## Tables + gated_full vs unet_drop verdict

In [ ]:
def ms(cfg, split, reg, key="iou"):
    a = np.array([r[split][reg][key] for r in records[cfg] if r[split][reg][key] is not None])
    return (float(a.mean()), float(a.std())) if len(a) else (float("nan"), 0.0)
def meanm(cfg, split):
    a = np.array([np.mean([r[split][reg]["iou"] for reg in regs]) for r in records[cfg]])
    return float(a.mean()), float(a.std())
def cell(t): return f"{t[0]:.3f}±{t[1]:.3f}"

order = [c for c in ["gated_full","gated_nofilm","gated_nocons","unet_drop","manet_drop","unet_naive"] if c in records]

for split, title in [("val", "VALIDATION"), ("bol", "BOLIVIA HOLDOUT")]:
    print("=" * 92)
    print(title)
    print(f"{'model':13s} {'both':>13s} {'s1_only':>13s} {'s2_only':>13s} {'mean':>13s}")
    print("-" * 92)
    for c in order:
        print(f"{c:13s} {cell(ms(c,split,'both')):>13s} {cell(ms(c,split,'s1_only')):>13s} "
              f"{cell(ms(c,split,'s2_only')):>13s} {cell(meanm(c,split)):>13s}")

print("=" * 92)
print("SAR-only robustness: gated_full vs unet_drop across splits")
print(f"{'metric':22s} {'gated_full':>13s} {'unet_drop':>13s} {'advantage':>11s}")
print("-" * 92)
if "gated_full" in records and "unet_drop" in records:
    for label, split, key in [("val SAR-only","val","iou"), ("val SAR flood-only","val","flood"),
                              ("bolivia SAR-only","bol","iou")]:
        g = ms("gated_full", split, "s1_only", key); u = ms("unet_drop", split, "s1_only", key)
        adv = g[0] - u[0]
        print(f"{label:22s} {cell(g):>13s} {cell(u):>13s} {adv:>+11.3f}")
    print("=" * 92)
    print("If the advantage is POSITIVE and LARGER on bolivia than on val,")
    print("the gated fusion + consistency wins where it matters (out-of-region robustness).")

json.dump({"records": records}, open("/kaggle/working/deep2.json","w"), indent=2, default=str)
print("saved deep2.json")